# Kaggle Training Notebook

Designed for **Save Version → Run All** (commit run). Do not run interactively cell-by-cell.

**What this notebook does:**
- Clones the repo, installs deps, prepares data (skipped if already done)
- Runs the checkpoint safety tests
- Trains transformer → TCN sequentially (~1.5h each, ~3h total)
- Zips all checkpoints to `/kaggle/working/checkpoints_transformer_tcn.zip` for download

**Mamba** is in `kaggle_train_mamba.ipynb` — its 23h runtime requires separate sessions.

**Before committing:**
- Accelerator → GPU T4 x1
- Internet ON (for git clone)
- Persistence ON
- Check quota ≥ 4 GB free

In [ ]:
# Cell 1: Clone repo and set working directory
import os, subprocess, sys

if not os.path.exists('transformer-vs-ssm'):
    subprocess.run(['git', 'clone',
                    'https://github.com/nvaidyan1/transformer-vs-ssm.git'], check=True)

os.chdir('transformer-vs-ssm')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 2: Install dependencies
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'torch', 'numpy', 'pyyaml', 'matplotlib', 'seaborn', 'tqdm'],
    check=True
)
print('deps OK')

In [ ]:
# Cell 3: Download and split enwik8 (skipped automatically if already done)
from src.data import prepare_data
prepare_data()

In [ ]:
# Cell 4: Pull latest commits
subprocess.run(['git', 'pull'], check=True)

In [ ]:
# Cell 5: Run checkpoint safety tests — must pass before training
result = subprocess.run(
    [sys.executable, 'tests/test_checkpoint_fixes.py'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-1000:])
assert result.returncode == 0, 'Tests failed — not starting training'

In [ ]:
# Cell 6: Verify disk headroom
import shutil
free_gb = shutil.disk_usage('/kaggle/working').free / 1024**3
print(f'Free disk: {free_gb:.1f} GB')
assert free_gb > 4.0, f'Need ≥4 GB free, got {free_gb:.1f} GB — clear space first'

In [ ]:
# Cell 7: Train transformer (~1.5h)
# -u: unbuffered stdout so log lines appear immediately in the output panel
print('=== TRANSFORMER TRAINING START ===')
result = subprocess.run(
    [sys.executable, '-u', 'src/train.py', '--config', 'configs/transformer.yaml'],
    check=False
)
print(f'=== TRANSFORMER EXIT CODE: {result.returncode} ===')
assert result.returncode == 0, 'Transformer training failed'

In [ ]:
# Cell 8: Verify transformer checkpoint
import torch
ckpt = torch.load('checkpoints/transformer/latest.pt', map_location='cpu', weights_only=False)
print(f'step:    {ckpt["step"]:,}')
print(f'val_bpc: {ckpt["val_bpc"]:.4f}')
assert ckpt['step'] == 50000, f'Expected 50000, got {ckpt["step"]}'
assert ckpt['val_bpc'] < 2.5, f'val_bpc too high: {ckpt["val_bpc"]:.4f}'
print('Transformer checkpoint OK')

In [ ]:
# Cell 9: Train TCN (~1.5h)
print('=== TCN TRAINING START ===')
result = subprocess.run(
    [sys.executable, '-u', 'src/train.py', '--config', 'configs/tcn.yaml'],
    check=False
)
print(f'=== TCN EXIT CODE: {result.returncode} ===')
assert result.returncode == 0, 'TCN training failed'

In [ ]:
# Cell 10: Verify TCN checkpoint
ckpt = torch.load('checkpoints/tcn/latest.pt', map_location='cpu', weights_only=False)
print(f'step:    {ckpt["step"]:,}')
print(f'val_bpc: {ckpt["val_bpc"]:.4f}')
assert ckpt['step'] == 50000
assert ckpt['val_bpc'] < 2.5
print('TCN checkpoint OK')

In [ ]:
# Cell 11: Zip transformer + TCN checkpoints as notebook output
import zipfile
from pathlib import Path

zip_path = Path('/kaggle/working/checkpoints_transformer_tcn.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for arch in ['transformer', 'tcn']:
        for f in sorted(Path(f'checkpoints/{arch}').rglob('*.pt')):
            zf.write(f)

size_mb = zip_path.stat().st_size / 1024**2
n_pts = sum(1 for _ in Path('checkpoints').rglob('*.pt'))
print(f'Zipped {n_pts} checkpoints → {zip_path} ({size_mb:.1f} MB)')
print('Download via: notebook Output tab → Download')